In [ ]:
# @title Install Missing Dependencies
!pip install groq pandas ipywidgets Pillow -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 30.7 MB/s eta 0:00:00


In [ ]:
# @title Setup & Core Functions
import os, re, json, base64, io, pandas as pd
from PIL import Image
from groq import Groq
import ipywidgets as widgets
from google.colab import userdata
from IPython.display import display, Image as IPyImage, clear_output

# Mount Drive & config
from google.colab import drive
drive.mount('/content/drive')
IMAGE_DIR = "/content/drive/MyDrive/CURLI"
CSV_PATH = os.path.join(IMAGE_DIR, ".results.csv")
API_KEY = userdata.get('API_KEY')
EMOTIONS = ["neutral","joy","sadness","anger","fear","disgust","surprise","hate","confusion","frustration","boredom","contempt"]
MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"

SYSTEM_PROMPT = f"""You are an ethical facial analysis assistant. Analyze the provided facial image and output ONLY a valid JSON object with the following structure. Do not include any other text, explanations, or markdown.

{{
  "emotions": {{
    "neutral": integer (0-100),
    "joy": integer (0-100),
    "sadness": integer (0-100),
    "anger": integer (0-100),
    "fear": integer (0-100),
    "disgust": integer (0-100),
    "surprise": integer (0-100),
    "hate": integer (0-100),
    "confusion": integer (0-100),
    "frustration": integer (0-100),
    "boredom": integer (0-100),
    "contempt": integer (0-100)
  }},
  "dominant_emotion": one of the 12 emotion keys above
}}

RULES:
1. All 12 emotion percentages MUST sum to exactly 100.
2. dominant_emotion must be the key with the highest integer value. If tie, pick the first alphabetically.
3. If confidence is low, distribute evenly or default to neutral — but still sum to 100.
4. Output ONLY the JSON object. No preamble, no code blocks, no explanations."""

def normalize_emotions(raw: dict) -> dict:
    vals = {e: max(0, min(100, int(round(float(raw.get(e,0)))))) for e in EMOTIONS}
    total = sum(vals.values())
    if total == 100: return vals
    if total <= 0: return {e: (100 if e=="neutral" else 0) for e in EMOTIONS}
    scaled = {e: vals[e]*100/total for e in EMOTIONS}
    floors = {e: int(scaled[e]) for e in EMOTIONS}
    rem = 100 - sum(floors.values())
    order = sorted(EMOTIONS, key=lambda k: (scaled[k]-floors[k]), reverse=True)
    for i in range(rem): floors[order[i%12]] += 1
    return floors

def parse_response(text: str) -> dict:
    try:
        match = re.search(r'\{.*\}', text.strip(), re.DOTALL)
        data = json.loads(match.group(0) if match else text)
        emotions = normalize_emotions(data.get("emotions", {}))
        dominant = min(EMOTIONS, key=lambda e: (-emotions[e], e))
        return {"emotions": emotions, "dominant_emotion": dominant}
    except: return {"emotions": {e:0 for e in EMOTIONS}, "dominant_emotion": "neutral"}

def get_ground_truth(filename: str) -> str:
    name = os.path.splitext(filename)[0].lower()
    if name.startswith("mixed"): return "unknown"
    for e in EMOTIONS:
        if name.startswith(e) and re.match(rf'^{e}\d+$', name): return e
    return "unknown"

def load_or_init_csv():
    cols = ["filename","ground_truth"]+EMOTIONS+["detected_emotion","ground_truth_pct","match_binary"]
    if os.path.exists(CSV_PATH):
        df = pd.read_csv(CSV_PATH)
        return df if list(df.columns)==cols else pd.DataFrame(columns=cols)
    return pd.DataFrame(columns=cols)

def save_row(filename, ground_truth, emotions, detected, gt_pct, match):
    df = load_or_init_csv()
    row = {"filename":filename,"ground_truth":ground_truth}|emotions|{"detected_emotion":detected,"ground_truth_pct":gt_pct,"match_binary":match}
    df = df[df["filename"]!=filename]  # overwrite if exists
    pd.concat([df, pd.DataFrame([row])], ignore_index=True).to_csv(CSV_PATH, index=False)

def analyze_image(filepath):
    with open(filepath, "rb") as f: img_bytes = f.read()
    encoded = base64.b64encode(img_bytes).decode()
    mime = "image/jpeg" if filepath.lower().endswith(".jpg") else "image/png"
    client = Groq(api_key=API_KEY)
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role":"system","content":SYSTEM_PROMPT},
            {"role":"user","content":[{"type":"text","text":"Analyze this facial image."},{"type":"image_url","image_url":{"url":f"data:{mime};base64,{encoded}"}}]}
        ],
        temperature=0.1, max_tokens=500, response_format={"type":"json_object"}
    )
    return parse_response(resp.choices[0].message.content or "")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# @title Batch Process All 100 Images -> Auto-Save CSV -> Show Final Table
if not os.path.isdir(IMAGE_DIR):
    raise FileNotFoundError(f"Drive path not found: {IMAGE_DIR}")

all_files = sorted([
    f for f in os.listdir(IMAGE_DIR)
    if os.path.isfile(os.path.join(IMAGE_DIR, f)) and f.lower().endswith(('.jpg','.jpeg','.png')) and not f.startswith('.')
], key=lambda x: (not x.startswith("mixed"), x.lower()))

print(f"Found {len(all_files)} images. Starting batch analysis...\n")

for i, fname in enumerate(all_files, 1):
    fpath = os.path.join(IMAGE_DIR, fname)
    if not os.path.exists(fpath):
        print(f"[{i}/{len(all_files)}] Skipped (not found on disk): {fname}")
        continue
    try:
        print(f"[{i}/{len(all_files)}] Analyzing: {fname} ...", end=" ")
        res = analyze_image(fpath)
        emotions = res["emotions"]
        detected = res["dominant_emotion"]
        gt = get_ground_truth(fname)
        gt_pct = emotions.get(gt, 0) if gt != "unknown" else 0
        match = 1 if (gt != "unknown" and detected == gt) else 0
        save_row(fname, gt, emotions, detected, gt_pct, match)
        print(f"Done: {detected} | Match: {'Yes' if match else 'No'}")
    except Exception as e:
        print(f"Error: {str(e)[:100]}")

# === FINAL DISPLAY ===
final_df = pd.read_csv(CSV_PATH).sort_values("filename").reset_index(drop=True)
print(f"\nSaved to: {CSV_PATH}")
labeled_total = len(final_df[final_df['ground_truth'] != 'unknown'])
match_total = int(final_df['match_binary'].sum())
print(f"Total rows: {len(final_df)} | Matches: {match_total} / {labeled_total}")
display(HTML("<h3>Final Results CSV</h3>"))
display(final_df)

Found 100 images. Starting batch analysis...

[1/100] Analyzing: mixed1.jpg ... Done: neutral | Match: No
[2/100] Analyzing: mixed2.jpg ... Done: confusion | Match: No
[3/100] Analyzing: mixed3.jpg ... Done: neutral | Match: No
[4/100] Analyzing: mixed4.jpg ... Done: neutral | Match: No
[5/100] Analyzing: anger1.jpg ... Done: anger | Match: Yes
[6/100] Analyzing: anger2.jpg ... Done: anger | Match: Yes
[7/100] Analyzing: anger3.jpg ... Done: anger | Match: Yes
[8/100] Analyzing: anger4.jpg ... Done: anger | Match: Yes
[9/100] Analyzing: anger5.jpg ... Done: anger | Match: Yes
[10/100] Analyzing: anger6.jpg ... Done: anger | Match: Yes
[11/100] Analyzing: anger7.jpg ... Done: joy | Match: No
[12/100] Analyzing: anger8.jpg ... Done: anger | Match: Yes
[13/100] Analyzing: boredom1.jpg ... Done: sadness | Match: No
[14/100] Analyzing: boredom2.jpg ... Done: sadness | Match: No
[15/100] Analyzing: boredom3.jpg ... Done: neutral | Match: No
[16/100] Analyzing: boredom4.jpg ... Done: neutral 

,filename,ground_truth,neutral,joy,sadness,anger,fear,disgust,surprise,hate,confusion,frustration,boredom,contempt,detected_emotion,ground_truth_pct,match_binary
0,anger1.jpg,anger,0,5,5,80,0,5,0,0,0,5,0,0,anger,80,1
1,anger2.jpg,anger,0,5,10,80,0,0,0,0,0,5,0,0,anger,80,1
2,anger3.jpg,anger,0,5,5,70,5,5,5,0,0,5,0,0,anger,70,1
3,anger4.jpg,anger,5,0,10,70,0,5,0,0,5,5,0,0,anger,70,1
4,anger5.jpg,anger,5,0,5,80,0,5,0,0,0,5,0,0,anger,80,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,sadness8.jpg,sadness,70,5,5,5,5,0,5,0,0,0,5,0,neutral,5,0
92,surpise1.jpg,unknown,5,70,5,0,5,0,10,0,0,0,0,5,joy,0,0
93,surpise2.jpg,unknown,9,18,5,5,14,5,36,0,4,0,0,4,surprise,0,0
94,surpise3.jpg,unknown,8,16,4,4,4,4,48,0,4,4,0,4,surprise,0,0


In [22]:
# @title Batch Process Remaining Images -> Auto-Save CSV -> Show Final Table

# Check for already processed files to skip them
processed_files = set()
if os.path.exists(CSV_PATH):
    try:
        df_exist = pd.read_csv(CSV_PATH)
        if "filename" in df_exist.columns:
            processed_files = set(df_exist["filename"].tolist())
    except Exception:
        pass

if not os.path.isdir(IMAGE_DIR):
    raise FileNotFoundError(f"Drive path not found: {IMAGE_DIR}")

all_files = sorted([
    f for f in os.listdir(IMAGE_DIR)
    if os.path.isfile(os.path.join(IMAGE_DIR, f)) and f.lower().endswith(('.jpg','.jpeg','.png')) and not f.startswith('.')
], key=lambda x: (not x.startswith("mixed"), x.lower()))

total_files = len(all_files)
remaining = total_files - len(processed_files)
print(f"Found {total_files} images. {len(processed_files)} already in CSV. Processing remaining {remaining}...\n")

for i, fname in enumerate(all_files, 1):
    if fname in processed_files:
        print(f"[{i}/{total_files}] Already processed. Skipping: {fname}")
        continue

    fpath = os.path.join(IMAGE_DIR, fname)
    if not os.path.exists(fpath):
        print(f"[{i}/{total_files}] Skipped (not found on disk): {fname}")
        continue
    try:
        print(f"[{i}/{total_files}] Analyzing: {fname} ...", end=" ")
        res = analyze_image(fpath)
        emotions = res["emotions"]
        detected = res["dominant_emotion"]
        gt = get_ground_truth(fname)
        gt_pct = emotions.get(gt, 0) if gt != "unknown" else 0
        match = 1 if (gt != "unknown" and detected == gt) else 0
        save_row(fname, gt, emotions, detected, gt_pct, match)
        print(f"Done: {detected} | Match: {'Yes' if match else 'No'}")
    except Exception as e:
        print(f"Error: {str(e)[:100]}")

# === FINAL DISPLAY ===
final_df = pd.read_csv(CSV_PATH).sort_values("filename").reset_index(drop=True)
print(f"\nSaved to: {CSV_PATH}")
labeled_total = len(final_df[final_df['ground_truth'] != 'unknown'])
match_total = int(final_df['match_binary'].sum())
print(f"Total rows: {len(final_df)} | Matches: {match_total} / {labeled_total}")
display(HTML("<h3>Final Results CSV</h3>"))
display(final_df)

Found 100 images. 92 already in CSV. Processing remaining 8...

[1/100] Already processed. Skipping: mixed1.jpg
[2/100] Already processed. Skipping: mixed2.jpg
[3/100] Already processed. Skipping: mixed3.jpg
[4/100] Already processed. Skipping: mixed4.jpg
[5/100] Already processed. Skipping: anger1.jpg
[6/100] Already processed. Skipping: anger2.jpg
[7/100] Already processed. Skipping: anger3.jpg
[8/100] Already processed. Skipping: anger4.jpg
[9/100] Already processed. Skipping: anger5.jpg
[10/100] Already processed. Skipping: anger6.jpg
[11/100] Already processed. Skipping: anger7.jpg
[12/100] Already processed. Skipping: anger8.jpg
[13/100] Already processed. Skipping: boredom1.jpg
[14/100] Already processed. Skipping: boredom2.jpg
[15/100] Already processed. Skipping: boredom3.jpg
[16/100] Already processed. Skipping: boredom4.jpg
[17/100] Already processed. Skipping: boredom5.jpg
[18/100] Already processed. Skipping: boredom6.jpg
[19/100] Already processed. Skipping: boredom7.jpg


,filename,ground_truth,neutral,joy,sadness,anger,fear,disgust,surprise,hate,confusion,frustration,boredom,contempt,detected_emotion,ground_truth_pct,match_binary
0,anger1.jpg,anger,0,5,5,80,0,5,0,0,0,5,0,0,anger,80,1
1,anger2.jpg,anger,0,5,10,80,0,0,0,0,0,5,0,0,anger,80,1
2,anger3.jpg,anger,0,5,5,70,5,5,5,0,0,5,0,0,anger,70,1
3,anger4.jpg,anger,5,0,10,70,0,5,0,0,5,5,0,0,anger,70,1
4,anger5.jpg,anger,5,0,5,80,0,5,0,0,0,5,0,0,anger,80,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,surprise4.jpg,surprise,10,70,5,0,0,0,10,0,0,0,0,5,joy,10,0
96,surprise5.jpg,surprise,5,80,0,5,5,0,5,0,0,0,0,0,joy,5,0
97,surprise6.jpg,surprise,10,20,5,5,5,5,40,0,5,0,0,5,surprise,40,1
98,surprise7.jpg,surprise,4,15,4,4,8,4,46,0,7,4,0,4,surprise,46,1


In [23]:
# @title Statistical Analysis — Per-Emotion Accuracy, Confusion Matrix, Hypothesis Test, Calibration

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import mannwhitneyu, pointbiserialr
from IPython.display import display, HTML

# ── CONFIG ────────────────────────────────────────────────────────────────────
CSV_PATH    = "/content/drive/MyDrive/CURLI/.results.csv"
OUTPUT_DIR  = "/content/drive/MyDrive/CURLI/Results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

EMOTIONS_12 = ["neutral","joy","sadness","anger","fear","disgust",
                "surprise","hate","confusion","frustration","boredom","contempt"]

BASIC_EMOTIONS   = ["neutral","joy","sadness","anger","fear","disgust","surprise"]
COMPLEX_EMOTIONS = ["hate","confusion","frustration","boredom","contempt"]

FIG_SIZE   = (10, 8)
FIG_DPI    = 300
CMAP       = "Blues"
CI_ALPHA   = 0.95

# ── HELPERS ───────────────────────────────────────────────────────────────────
def clopper_pearson(k, n, alpha=0.05):
    """Exact binomial CI. Returns (lower, upper)."""
    if n == 0:
        return (0.0, 1.0)
    lo = stats.beta.ppf(alpha / 2,     k,     n - k + 1) if k > 0 else 0.0
    hi = stats.beta.ppf(1 - alpha / 2, k + 1, n - k)     if k < n else 1.0
    return (lo, hi)

def save_fig(fig, name):
    path = os.path.join(OUTPUT_DIR, name)
    fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
    print(f"  Saved → {path}")

def save_csv(df, name):
    path = os.path.join(OUTPUT_DIR, name)
    df.to_csv(path, index=False)
    print(f"  Saved → {path}")

def section(title):
    print(f"\n{'='*70}\n  {title}\n{'='*70}")

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
section("Loading data")
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"Results CSV not found: {CSV_PATH}")

df = pd.read_csv(CSV_PATH)
required = {"filename","ground_truth","detected_emotion","ground_truth_pct","match_binary"}
missing  = required - set(df.columns)
if missing:
    raise ValueError(f"CSV is missing columns: {missing}")

print(f"  Rows loaded : {len(df)}")
print(f"  Columns     : {list(df.columns)}")

# Separate mixed / unknown rows — excluded from accuracy + hypothesis, kept for confusion
df_labeled = df[df["ground_truth"].isin(EMOTIONS_12)].copy()
df_mixed   = df[~df["ground_truth"].isin(EMOTIONS_12)].copy()
print(f"  Labeled rows (12 emotions) : {len(df_labeled)}")
print(f"  Mixed / unknown rows       : {len(df_mixed)}")

summary_lines = []   # accumulate text for final report

# ══════════════════════════════════════════════════════════════════════════════
# ANALYSIS 1 — Per-Emotion Accuracy + Clopper-Pearson 95% CI
# ══════════════════════════════════════════════════════════════════════════════
section("Analysis 1 · Per-Emotion Accuracy + Clopper-Pearson 95% CI")

emotion_categories = EMOTIONS_12 + (["mixed"] if len(df_mixed) > 0 else [])
rows_acc = []

for emotion in emotion_categories:
    if emotion == "mixed":
        sub = df_mixed
        k   = int(sub["match_binary"].sum()) if "match_binary" in sub.columns else 0
    else:
        sub = df_labeled[df_labeled["ground_truth"] == emotion]
        k   = int(sub["match_binary"].sum())
    n   = len(sub)
    acc = k / n if n > 0 else np.nan
    lo, hi = clopper_pearson(k, n)
    rows_acc.append({
        "emotion"   : emotion,
        "n"         : n,
        "correct"   : k,
        "accuracy"  : round(acc, 4) if not np.isnan(acc) else np.nan,
        "ci_lower"  : round(lo, 4),
        "ci_upper"  : round(hi, 4),
        "ci_width"  : round(hi - lo, 4),
    })

df_acc = pd.DataFrame(rows_acc)
display(HTML("<h4>Per-Emotion Accuracy with 95% Clopper-Pearson CI</h4>"))
display(df_acc)
save_csv(df_acc, "per_emotion_accuracy.csv")

# Bar chart with error bars
fig, ax = plt.subplots(figsize=FIG_SIZE)
valid   = df_acc.dropna(subset=["accuracy"])
colors  = ["#4393c3" if e in BASIC_EMOTIONS else
           "#d6604d" if e in COMPLEX_EMOTIONS else
           "#878787" for e in valid["emotion"]]

bars = ax.bar(valid["emotion"], valid["accuracy"], color=colors,
              edgecolor="white", linewidth=0.8, zorder=3)
ax.errorbar(
    x          = range(len(valid)),
    y          = valid["accuracy"],
    yerr       = [valid["accuracy"] - valid["ci_lower"],
                  valid["ci_upper"] - valid["accuracy"]],
    fmt        = "none",
    ecolor     = "black",
    elinewidth = 1.5,
    capsize    = 5,
    capthick   = 1.5,
    zorder     = 4,
)
ax.axhline(0.5, color="gray", linewidth=0.8, linestyle="--", alpha=0.6)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_xlabel("Ground-Truth Emotion", fontsize=12)
ax.set_title("Per-Emotion Classification Accuracy\nwith 95% Clopper-Pearson Confidence Intervals", fontsize=13)
ax.set_xticks(range(len(valid)))
ax.set_xticklabels(valid["emotion"], rotation=40, ha="right", fontsize=10)
ax.grid(axis="y", alpha=0.3, zorder=0)
legend_patches = [
    mpatches.Patch(color="#4393c3", label="Basic emotions"),
    mpatches.Patch(color="#d6604d", label="Complex emotions"),
    mpatches.Patch(color="#878787", label="Mixed"),
]
ax.legend(handles=legend_patches, fontsize=10)
plt.tight_layout()
plt.show()
save_fig(fig, "accuracy_with_ci.png")
plt.close(fig)

summary_lines.append("=== Analysis 1: Per-Emotion Accuracy ===")
for _, r in df_acc.iterrows():
    summary_lines.append(
        f"  {r['emotion']:15s}  n={r['n']:3d}  acc={r['accuracy']:.3f}"
        f"  95% CI [{r['ci_lower']:.3f}, {r['ci_upper']:.3f}]"
    )

# ══════════════════════════════════════════════════════════════════════════════
# ANALYSIS 2 — 13×12 Confusion Matrix + Classification Metrics
# ══════════════════════════════════════════════════════════════════════════════
section("Analysis 2 · Confusion Matrix + Classification Metrics")

gt_categories  = EMOTIONS_12 + (["mixed"] if len(df_mixed) > 0 else [])
det_categories = EMOTIONS_12   # model cannot output "mixed"

# Build confusion matrix
conf_counts = pd.DataFrame(0, index=gt_categories, columns=det_categories)

for _, row in df_labeled.iterrows():
    gt  = row["ground_truth"]
    det = row["detected_emotion"]
    if gt in conf_counts.index and det in conf_counts.columns:
        conf_counts.loc[gt, det] += 1

if len(df_mixed) > 0:
    for _, row in df_mixed.iterrows():
        det = row["detected_emotion"]
        if det in conf_counts.columns:
            conf_counts.loc["mixed", det] += 1

# Normalized (row-wise)
row_sums = conf_counts.sum(axis=1).replace(0, np.nan)
conf_norm = conf_counts.div(row_sums, axis=0).fillna(0).round(3)

display(HTML("<h4>Confusion Matrix (Raw Counts)</h4>"))
display(conf_counts)
save_csv(conf_counts.reset_index().rename(columns={"index":"ground_truth"}),
         "confusion_matrix_counts.csv")

display(HTML("<h4>Confusion Matrix (Row-Normalized)</h4>"))
display(conf_norm)
save_csv(conf_norm.reset_index().rename(columns={"index":"ground_truth"}),
         "confusion_matrix_normalized.csv")

def plot_confusion(data, title, fname, fmt):
    fig, ax = plt.subplots(figsize=FIG_SIZE)
    sns.heatmap(data, annot=True, fmt=fmt, cmap=CMAP,
                linewidths=0.4, linecolor="white",
                cbar_kws={"shrink": 0.8}, ax=ax)
    ax.set_title(title, fontsize=13, pad=12)
    ax.set_xlabel("Detected Emotion", fontsize=11)
    ax.set_ylabel("Ground-Truth Emotion", fontsize=11)
    ax.tick_params(axis="x", rotation=40, labelsize=9)
    ax.tick_params(axis="y", rotation=0,  labelsize=9)
    plt.tight_layout()
    plt.show()
    save_fig(fig, fname)
    plt.close(fig)

plot_confusion(conf_counts, "Confusion Matrix — Raw Counts",
               "confusion_raw.png", "d")
plot_confusion(conf_norm,   "Confusion Matrix — Row-Normalized (Recall)",
               "confusion_normalized.png", ".2f")

# Per-class metrics (only for the 12 labeled emotion rows)
metrics_rows = []
for emotion in EMOTIONS_12:
    if emotion not in conf_counts.index:
        continue
    TP = conf_counts.loc[emotion, emotion] if emotion in conf_counts.columns else 0
    FP = conf_counts[emotion].sum() - TP if emotion in conf_counts.columns else 0
    FN = conf_counts.loc[emotion].sum() - TP
    precision = TP / (TP + FP) if (TP + FP) > 0 else np.nan
    recall    = TP / (TP + FN) if (TP + FN) > 0 else np.nan
    f1        = (2 * precision * recall / (precision + recall)
                 if not (np.isnan(precision) or np.isnan(recall) or (precision + recall) == 0)
                 else np.nan)
    metrics_rows.append({
        "emotion"  : emotion,
        "TP"       : int(TP),
        "FP"       : int(FP),
        "FN"       : int(FN),
        "precision": round(precision, 4) if not np.isnan(precision) else np.nan,
        "recall"   : round(recall,    4) if not np.isnan(recall)    else np.nan,
        "f1_score" : round(f1,        4) if not np.isnan(f1)        else np.nan,
        "support"  : int(conf_counts.loc[emotion].sum()),
    })

df_metrics = pd.DataFrame(metrics_rows)
display(HTML("<h4>Classification Metrics per Emotion</h4>"))
display(df_metrics)
save_csv(df_metrics, "classification_metrics.csv")

summary_lines.append("\n=== Analysis 2: Classification Metrics ===")
for _, r in df_metrics.iterrows():
    summary_lines.append(
        f"  {r['emotion']:15s}  P={r['precision']:.3f}  R={r['recall']:.3f}  F1={r['f1_score']:.3f}"
    )

# ══════════════════════════════════════════════════════════════════════════════
# ANALYSIS 3 — Basic vs. Complex Hypothesis Test (Mann-Whitney U)
# ══════════════════════════════════════════════════════════════════════════════
section("Analysis 3 · Basic vs. Complex Emotion Hypothesis Test (Mann-Whitney U)")

basic_scores   = df_labeled[df_labeled["ground_truth"].isin(BASIC_EMOTIONS)]["match_binary"].values
complex_scores = df_labeled[df_labeled["ground_truth"].isin(COMPLEX_EMOTIONS)]["match_binary"].values

print(f"  Basic   group : n={len(basic_scores)},  mean={basic_scores.mean():.3f}")
print(f"  Complex group : n={len(complex_scores)}, mean={complex_scores.mean():.3f}")

U_stat, p_value = mannwhitneyu(basic_scores, complex_scores, alternative="two-sided")
n1, n2          = len(basic_scores), len(complex_scores)
rank_biserial   = 1 - (2 * U_stat) / (n1 * n2)   # r = 1 - 2U/(n1*n2)

print(f"\n  Mann-Whitney U statistic : {U_stat:.4f}")
print(f"  Two-tailed p-value       : {p_value:.4f}")
print(f"  Rank-biserial correlation: {rank_biserial:.4f}")
print(f"  Interpretation: {'Significant (p < 0.05)' if p_value < 0.05 else 'Not significant (p ≥ 0.05)'}")

df_hyp = pd.DataFrame([{
    "group_basic_n"           : n1,
    "group_complex_n"         : n2,
    "basic_mean_accuracy"     : round(basic_scores.mean(),   4),
    "complex_mean_accuracy"   : round(complex_scores.mean(), 4),
    "mann_whitney_U"          : round(U_stat,        4),
    "p_value_two_tailed"      : round(p_value,        6),
    "rank_biserial_r"         : round(rank_biserial,  4),
    "significant_alpha_0.05"  : p_value < 0.05,
}])
display(df_hyp)
save_csv(df_hyp, "hypothesis_test_results.csv")

# Box plot
fig, ax = plt.subplots(figsize=(7, 6))
plot_data  = [basic_scores, complex_scores]
plot_labels = [f"Basic\n(n={n1})", f"Complex\n(n={n2})"]
bp = ax.boxplot(plot_data, labels=plot_labels, patch_artist=True,
                medianprops=dict(color="black", linewidth=2),
                whiskerprops=dict(linewidth=1.5),
                capprops=dict(linewidth=1.5))
colors_box = ["#4393c3", "#d6604d"]
for patch, color in zip(bp["boxes"], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
# Overlay jittered points
for i, (vals, color) in enumerate(zip(plot_data, colors_box), start=1):
    jitter = np.random.default_rng(42).uniform(-0.08, 0.08, size=len(vals))
    ax.scatter(i + jitter, vals, alpha=0.5, color=color, s=30, zorder=3)
sig_text = f"Mann-Whitney U={U_stat:.1f}, p={p_value:.4f}\nr={rank_biserial:.3f}"
ax.text(0.97, 0.97, sig_text, transform=ax.transAxes,
        ha="right", va="top", fontsize=10,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="gray"))
ax.set_ylim(-0.2, 1.3)
ax.set_ylabel("match_binary (0 = wrong, 1 = correct)", fontsize=11)
ax.set_title("Classification Accuracy:\nBasic vs. Complex Emotions", fontsize=13)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()
save_fig(fig, "accuracy_by_group.png")
plt.close(fig)

summary_lines.append("\n=== Analysis 3: Hypothesis Test (Mann-Whitney U) ===")
summary_lines.append(f"  Basic   mean accuracy : {basic_scores.mean():.3f}  (n={n1})")
summary_lines.append(f"  Complex mean accuracy : {complex_scores.mean():.3f}  (n={n2})")
summary_lines.append(f"  U = {U_stat:.4f}, p = {p_value:.6f}, r = {rank_biserial:.4f}")
summary_lines.append(f"  Result: {'SIGNIFICANT — basic emotions classified more accurately' if (p_value < 0.05 and rank_biserial > 0) else 'Not significant at alpha=0.05'}")

# ══════════════════════════════════════════════════════════════════════════════
# ANALYSIS 4 — Confidence Calibration (Point-Biserial Correlation)
# ══════════════════════════════════════════════════════════════════════════════
section("Analysis 4 · Confidence Calibration Analysis")

calib_rows = []

for emotion in EMOTIONS_12:
    sub = df_labeled[df_labeled["ground_truth"] == emotion]
    if len(sub) < 3:
        calib_rows.append({
            "emotion": emotion, "n": len(sub),
            "r_pb": np.nan, "p_value": np.nan, "note": "insufficient data"
        })
        continue
    conf   = sub["ground_truth_pct"].values.astype(float)
    binary = sub["match_binary"].values.astype(float)
    if binary.std() == 0:
        calib_rows.append({
            "emotion": emotion, "n": len(sub),
            "r_pb": np.nan, "p_value": np.nan, "note": "no variance in match_binary"
        })
        continue
    r_pb, p_val = pointbiserialr(binary, conf)
    calib_rows.append({
        "emotion" : emotion,
        "n"       : len(sub),
        "r_pb"    : round(r_pb,  4),
        "p_value" : round(p_val, 6),
        "note"    : "significant" if p_val < 0.05 else "not significant",
    })

df_calib = pd.DataFrame(calib_rows)
display(HTML("<h4>Confidence Calibration — Point-Biserial Correlation</h4>"))
display(df_calib)
save_csv(df_calib, "confidence_calibration.csv")

# Scatter/regression plot — one panel per emotion
valid_calib = [e for e in EMOTIONS_12
               if not df_calib[df_calib["emotion"] == e]["r_pb"].isna().all()]
n_panels = len(valid_calib)
ncols    = 4
nrows    = int(np.ceil(n_panels / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 3.2))
axes_flat = axes.flatten() if n_panels > 1 else [axes]

palette = {"Basic": "#4393c3", "Complex": "#d6604d"}

for idx, emotion in enumerate(valid_calib):
    ax  = axes_flat[idx]
    sub = df_labeled[df_labeled["ground_truth"] == emotion]
    grp = "Basic" if emotion in BASIC_EMOTIONS else "Complex"
    color = palette[grp]

    x = sub["ground_truth_pct"].values.astype(float)
    y = sub["match_binary"].values.astype(float)

    ax.scatter(x, y + np.random.default_rng(idx).uniform(-0.04, 0.04, len(y)),
               alpha=0.6, color=color, s=40, edgecolors="white", linewidth=0.5)

    # Regression line
    if len(x) > 1 and x.std() > 0:
        m, b = np.polyfit(x, y, 1)
        x_line = np.linspace(x.min(), x.max(), 100)
        ax.plot(x_line, m * x_line + b, color=color, linewidth=1.8, linestyle="--")

    row_c = df_calib[df_calib["emotion"] == emotion].iloc[0]
    r_str = f"r={row_c['r_pb']:.2f}, p={row_c['p_value']:.3f}" if not np.isnan(row_c["r_pb"]) else "n/a"
    ax.set_title(f"{emotion}\n{r_str}", fontsize=9)
    ax.set_xlabel("Model Confidence (%)", fontsize=8)
    ax.set_ylabel("Correct (1) / Wrong (0)", fontsize=8)
    ax.set_ylim(-0.25, 1.25)
    ax.set_yticks([0, 1])
    ax.tick_params(labelsize=7)
    ax.grid(alpha=0.2)

# Hide unused panels
for idx in range(n_panels, len(axes_flat)):
    axes_flat[idx].set_visible(False)

legend_patches = [mpatches.Patch(color=v, label=k) for k, v in palette.items()]
fig.legend(handles=legend_patches, loc="lower right", fontsize=10)
fig.suptitle("Confidence Calibration: Model Confidence vs. Classification Correctness\n"
             "(Point-Biserial Correlation per Emotion)", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()
save_fig(fig, "confidence_correlation.png")
plt.close(fig)

summary_lines.append("\n=== Analysis 4: Confidence Calibration ===")
for _, r in df_calib.iterrows():
    if not np.isnan(r["r_pb"]):
        summary_lines.append(
            f"  {r['emotion']:15s}  r_pb={r['r_pb']:.3f}  p={r['p_value']:.4f}  ({r['note']})"
        )
    else:
        summary_lines.append(f"  {r['emotion']:15s}  {r['note']}")

# ══════════════════════════════════════════════════════════════════════════════
# WRITE SUMMARY REPORT
# ══════════════════════════════════════════════════════════════════════════════
section("Writing analysis_summary.txt")

header = [
    "EMOTION RECOGNITION STATISTICAL ANALYSIS REPORT",
    f"Generated from: {CSV_PATH}",
    f"Total rows: {len(df)} | Labeled: {len(df_labeled)} | Mixed/unknown: {len(df_mixed)}",
    "",
]
report_text = "\n".join(header + summary_lines)

report_path = os.path.join(OUTPUT_DIR, "analysis_summary.txt")
with open(report_path, "w") as f:
    f.write(report_text)
print(f"  Saved → {report_path}")
print("\n" + report_text)

# ══════════════════════════════════════════════════════════════════════════════
section("All analyses complete")
print(f"  Output directory: {OUTPUT_DIR}")
print("  Files written:")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    size  = os.path.getsize(fpath)
    print(f"    {fname:<45s}  {size:>8,} bytes")


  Loading data
  Rows loaded : 100
  Columns     : ['filename', 'ground_truth', 'neutral', 'joy', 'sadness', 'anger', 'fear', 'disgust', 'surprise', 'hate', 'confusion', 'frustration', 'boredom', 'contempt', 'detected_emotion', 'ground_truth_pct', 'match_binary']
  Labeled rows (12 emotions) : 96
  Mixed / unknown rows       : 4

  Analysis 1 · Per-Emotion Accuracy + Clopper-Pearson 95% CI


,emotion,n,correct,accuracy,ci_lower,ci_upper,ci_width
0,neutral,8,8,1.000,0.6306,1.0000,0.3694
1,joy,8,8,1.000,0.6306,1.0000,0.3694
2,sadness,8,5,0.625,0.2449,0.9148,0.6699
3,anger,8,7,0.875,0.4735,0.9968,0.5233
4,fear,8,6,0.750,0.3491,0.9681,0.6190
5,disgust,8,2,0.250,0.0319,0.6509,0.6190
6,surprise,8,5,0.625,0.2449,0.9148,0.6699
7,hate,8,0,0.000,0.0000,0.3694,0.3694
8,confusion,8,4,0.500,0.1570,0.8430,0.6860
9,frustration,8,1,0.125,0.0032,0.5265,0.5233


  Saved → /content/drive/MyDrive/CURLI/Results/per_emotion_accuracy.csv
  Saved → /content/drive/MyDrive/CURLI/Results/accuracy_with_ci.png

  Analysis 2 · Confusion Matrix + Classification Metrics


,neutral,joy,sadness,anger,fear,disgust,surprise,hate,confusion,frustration,boredom,contempt
neutral,8,0,0,0,0,0,0,0,0,0,0,0
joy,0,8,0,0,0,0,0,0,0,0,0,0
sadness,2,0,5,1,0,0,0,0,0,0,0,0
anger,0,1,0,7,0,0,0,0,0,0,0,0
fear,1,0,0,0,6,0,1,0,0,0,0,0
disgust,1,0,1,3,0,2,1,0,0,0,0,0
surprise,0,3,0,0,0,0,5,0,0,0,0,0
hate,1,0,2,4,0,0,1,0,0,0,0,0
confusion,0,1,0,1,0,0,2,0,4,0,0,0
frustration,1,0,0,4,0,0,1,0,1,1,0,0


  Saved → /content/drive/MyDrive/CURLI/Results/confusion_matrix_counts.csv


...

  Saved → /content/drive/MyDrive/CURLI/Results/hypothesis_test_results.csv


/tmp/ipykernel_13183/3588450681.py:284: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(plot_data, labels=plot_labels, patch_artist=True,


  Saved → /content/drive/MyDrive/CURLI/Results/accuracy_by_group.png

  Analysis 4 · Confidence Calibration Analysis


,emotion,n,r_pb,p_value,note
0,neutral,8,NaN,NaN,no variance in match_binary
1,joy,8,NaN,NaN,no variance in match_binary
2,sadness,8,0.9332,0.000708,significant
3,anger,8,0.9406,0.000500,significant
4,fear,8,0.8088,0.015062,significant
5,disgust,8,0.8859,0.003404,significant
6,surprise,8,0.9625,0.000128,significant
7,hate,8,NaN,NaN,no variance in match_binary
8,confusion,8,0.8929,0.002827,significant
9,frustration,8,0.4651,0.245487,not significant


  Saved → /content/drive/MyDrive/CURLI/Results/confidence_calibration.csv
  Saved → /content/drive/MyDrive/CURLI/Results/confidence_correlation.png

  Writing analysis_summary.txt
  Saved → /content/drive/MyDrive/CURLI/Results/analysis_summary.txt

EMOTION RECOGNITION STATISTICAL ANALYSIS REPORT
Generated from: /content/drive/MyDrive/CURLI/.results.csv
Total rows: 100 | Labeled: 96 | Mixed/unknown: 4

=== Analysis 1: Per-Emotion Accuracy ===
  neutral          n=  8  acc=1.000  95% CI [0.631, 1.000]
  joy              n=  8  acc=1.000  95% CI [0.631, 1.000]
  sadness          n=  8  acc=0.625  95% CI [0.245, 0.915]
  anger            n=  8  acc=0.875  95% CI [0.473, 0.997]
  fear             n=  8  acc=0.750  95% CI [0.349, 0.968]
  disgust          n=  8  acc=0.250  95% CI [0.032, 0.651]
  surprise         n=  8  acc=0.625  95% CI [0.245, 0.915]
  hate             n=  8  acc=0.000  95% CI [0.000, 0.369]
  confusion        n=  8  acc=0.500  95% CI [0.157, 0.843]
  frustration      n=  

In [24]:
# @title Statistical Analysis Plan
import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import binomtest, mannwhitneyu, pointbiserialr

warnings.filterwarnings('ignore')

try:
    # --- CONFIG & PATHS ---
    BASE_DIR = "/content/drive/MyDrive/CURLI"
    CSV_PATH = os.path.join(BASE_DIR, ".results.csv")
    OUT_DIR = os.path.join(BASE_DIR, "Result")
    os.makedirs(OUT_DIR, exist_ok=True)

    # Load data
    df = pd.read_csv(CSV_PATH)

    # Standardize ground truth labels (map 'unknown' from mixed files to 'mixed')
    df['gt_clean'] = df['ground_truth'].copy()
    mask_mixed = df['filename'].str.lower().str.startswith('mixed')
    df.loc[mask_mixed, 'gt_clean'] = 'mixed'

    # Keep only valid GT categories
    VALID_GT = ["neutral","joy","sadness","anger","fear","disgust","surprise",
                "hate","confusion","frustration","boredom","contempt","mixed"]
    df = df[df['gt_clean'].isin(VALID_GT)].copy()
    df['gt_clean'] = pd.Categorical(df['gt_clean'], categories=VALID_GT, ordered=False)

    EMOTIONS = ["neutral","joy","sadness","anger","fear","disgust","surprise",
                "hate","confusion","frustration","boredom","contempt"]
    GT_CLASSES = EMOTIONS + ["mixed"]
    PRED_CLASSES = EMOTIONS

    # Define hypothesis groups (mixed excluded per spec)
    BASIC = ["neutral", "joy", "sadness", "anger", "fear", "disgust", "surprise"]
    COMPLEX = ["hate", "confusion", "frustration", "boredom", "contempt"]

    print(f"Loaded {len(df)} valid samples. Starting statistical analysis...")

    # ==========================================
    # 1. PER-EMOTION ACCURACY + CLOPPER-PEARSON 95% CI
    # ==========================================
    acc_data = []
    for gt in GT_CLASSES:
        sub = df[df['gt_clean'] == gt]
        n = len(sub)
        k = int(sub['match_binary'].sum())
        if n > 0:
            bt = binomtest(k, n, alternative='two-sided')
            ci = bt.proportion_ci(confidence_level=0.95)
            acc_data.append({
                'ground_truth': gt, 'matches': k, 'total': n,
                'accuracy': k/n, 'ci_low': ci.low, 'ci_high': ci.high
            })
    acc_df = pd.DataFrame(acc_data)
    acc_df.to_csv(os.path.join(OUT_DIR, 'per_emotion_accuracy.csv'), index=False)

    plt.figure(figsize=(10, 8))
    ax = sns.barplot(x='ground_truth', y='accuracy', data=acc_df, color='#1f77b4', edgecolor='black')
    x_pos = np.arange(len(acc_df))
    plt.errorbar(x_pos, acc_df['accuracy'],
                 yerr=[acc_df['accuracy'] - acc_df['ci_low'], acc_df['ci_high'] - acc_df['accuracy']],
                 fmt='none', ecolor='black', capsize=5, lw=1.5)
    plt.xticks(x_pos, acc_df['ground_truth'], rotation=45, ha='right')
    plt.ylim(0, 1.15)
    plt.ylabel('Accuracy')
    plt.xlabel('Ground Truth Emotion')
    plt.title('Per-Emotion Accuracy with 95% Clopper-Pearson CI')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'accuracy_with_ci.png'), dpi=300)
    plt.show()
    print("Saved: per_emotion_accuracy.csv & accuracy_with_ci.png")
    plt.close()

    # ==========================================
    # 2. 13x12 CONFUSION MATRIX + CLASSIFICATION METRICS
    # ==========================================
    cm = pd.crosstab(df['gt_clean'], df['detected_emotion'], rownames=['Ground Truth'], colnames=['Predicted'])
    cm = cm.reindex(index=GT_CLASSES, columns=PRED_CLASSES, fill_value=0)
    cm_norm = cm.div(cm.sum(axis=1), axis=0).fillna(0)

    cm.to_csv(os.path.join(OUT_DIR, 'confusion_matrix_counts.csv'))
    cm_norm.to_csv(os.path.join(OUT_DIR, 'confusion_matrix_normalized.csv'))

    # Raw CM
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, linewidths=0.5)
    plt.title('Confusion Matrix (Counts)')
    plt.ylabel('Ground Truth')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'confusion_raw.png'), dpi=300)
    plt.show()
    print("Saved: confusion_raw.png")
    plt.close()

    # Normalized CM
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', cbar=False, linewidths=0.5)
    plt.title('Confusion Matrix (Row-Normalized)')
    plt.ylabel('Ground Truth')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'confusion_normalized.png'), dpi=300)
    plt.show()
    print("Saved: confusion_normalized.png")
    plt.close()

    # Metrics per GT row
    metrics = []
    for gt in GT_CLASSES:
        tp = cm.loc[gt, gt] if gt in cm.columns else 0
        fn = cm.loc[gt].sum() - tp
        fp = cm[gt].sum() - tp if gt in cm.columns else 0
        support = tp + fn
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / support if support > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        metrics.append({'ground_truth': gt, 'precision': round(precision, 4),
                        'recall': round(recall, 4), 'f1': round(f1, 4), 'support': support})
    pd.DataFrame(metrics).to_csv(os.path.join(OUT_DIR, 'classification_metrics.csv'), index=False)
    print("Saved: classification_metrics.csv")

    # ==========================================
    # 3. BASIC VS. COMPLEX HYPOTHESIS TEST
    # ==========================================
    basic_data = df[df['gt_clean'].isin(BASIC)]['match_binary']
    complex_data = df[df['gt_clean'].isin(COMPLEX)]['match_binary']

    if len(basic_data) > 0 and len(complex_data) > 0:
        u_stat, p_val = mannwhitneyu(basic_data, complex_data, alternative='two-sided')
        n1, n2 = len(basic_data), len(complex_data)
        r_rb = 1 - (2 * u_stat) / (n1 * n2)
        interp = 'Significant' if p_val < 0.05 else 'Not Significant'
    else:
        u_stat, p_val, r_rb, n1, n2, interp = np.nan, np.nan, np.nan, 0, 0, 'Insufficient Data'

    pd.DataFrame([{
        'test': 'Mann-Whitney U', 'U_statistic': u_stat, 'p_value': p_val,
        'n_basic': n1, 'n_complex': n2, 'rank_biserial_r': r_rb, 'interpretation': interp
    }]).to_csv(os.path.join(OUT_DIR, 'hypothesis_test_results.csv'), index=False)

    plot_df = df[df['gt_clean'].isin(BASIC + COMPLEX)].copy()
    plot_df['group'] = plot_df['gt_clean'].apply(lambda x: 'BASIC' if x in BASIC else 'COMPLEX')
    plt.figure(figsize=(10, 8))
    sns.boxplot(x='group', y='match_binary', data=plot_df, palette=['#1f77b4', '#ff7f0e'], width=0.4)
    plt.title('Accuracy Distribution: Basic vs. Complex Emotions')
    plt.ylabel('Match Binary (0=Incorrect, 1=Correct)')
    plt.ylim(-0.15, 1.15)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'accuracy_by_group.png'), dpi=300)
    plt.show()
    print("Saved: hypothesis_test_results.csv & accuracy_by_group.png")
    plt.close()

    # ==========================================
    # 4. CONFIDENCE CALIBRATION ANALYSIS
    # ==========================================
    calib_data = []
    for gt in GT_CLASSES:
        sub = df[df['gt_clean'] == gt]
        if len(sub) >= 2:
            r, p = pointbiserialr(sub['ground_truth_pct'], sub['match_binary'])
            calib_data.append({'ground_truth': gt, 'point_biserial_r': round(r, 4), 'p_value': round(p, 4), 'n': len(sub)})
        else:
            calib_data.append({'ground_truth': gt, 'point_biserial_r': np.nan, 'p_value': np.nan, 'n': len(sub)})
    pd.DataFrame(calib_data).to_csv(os.path.join(OUT_DIR, 'confidence_calibration.csv'), index=False)

    plot_calib = df.dropna(subset=['ground_truth_pct', 'match_binary']).copy()
    plot_calib['group'] = plot_calib['gt_clean'].apply(lambda x: 'BASIC' if x in BASIC else ('COMPLEX' if x in COMPLEX else 'MIXED'))
    g = sns.lmplot(x='ground_truth_pct', y='match_binary', hue='group', data=plot_calib,
                   height=8, aspect=1.25, scatter_kws={'alpha':0.7, 's':60},
                   line_kws={'lw':2.5}, ci=95)
    g.fig.suptitle('Confidence Calibration: Confidence vs. Correctness by Group', y=1.02, fontsize=14)
    g.set_axis_labels('Model Confidence for Ground Truth (%)', 'Match Binary (Correct=1)')
    g.set(ylim=(-0.1, 1.1), xlim=(0, 100))
    g.fig.tight_layout()
    g.fig.savefig(os.path.join(OUT_DIR, 'confidence_correlation.png'), dpi=300)
    plt.show()
    print("Saved: confidence_calibration.csv & confidence_correlation.png")
    plt.close()

    # ==========================================
    # SUMMARY REPORT
    # ==========================================
    overall_acc = df['match_binary'].mean()
    sig_calib = [c['ground_truth'] for c in calib_data if not np.isnan(c.get('p_value', np.nan)) and c['p_value'] < 0.05]

    summary_text = f"""=== STATISTICAL ANALYSIS SUMMARY ===\nDataset Size: {len(df)} valid samples\nOverall Accuracy: {overall_acc:.3f}\n\n--- Per-Emotion Accuracy ---\n{acc_df[['ground_truth', 'accuracy', 'ci_low', 'ci_high']].to_string(index=False)}\n\n--- Hypothesis Test (Basic vs. Complex) ---\nMann-Whitney U = {u_stat:.3f}, p = {p_val:.4f}\nRank-Biserial Correlation (r) = {r_rb:.3f}\nInterpretation: {'Basic emotions were classified significantly more accurately than complex ones.' if p_val < 0.05 else 'No significant difference found between basic and complex emotion classification accuracy.'}\n\n--- Confidence Calibration ---\nSignificant correlations (p<0.05) found for: {', '.join(sig_calib) if sig_calib else 'None'}\nAverage point-biserial r across classes: {np.nanmean([c['point_biserial_r'] for c in calib_data]):.3f}\n\n--- Notes ---\n1. Clopper-Pearson exact intervals used for binomial accuracy.\n2. Confusion matrix is 13x12 (GT includes 'mixed', model cannot predict it).\n3. Basic/Complex grouping follows standard FER literature (7 vs 5).\n4. All figures saved at 300 DPI, 10x8 inches.\n5. Analysis completed successfully without errors."""

    with open(os.path.join(OUT_DIR, 'analysis_summary.txt'), 'w') as f:
        f.write(summary_text)
    print("Saved: analysis_summary.txt")
    print(summary_text)

except Exception as e:
    print(f"CRITICAL ERROR: {e}")
    import traceback
    traceback.print_exc()
    print("Execution halted. Please review the traceback above.")

Loaded 100 valid samples. Starting statistical analysis...
Saved: per_emotion_accuracy.csv & accuracy_with_ci.png
Saved: confusion_raw.png
Saved: confusion_normalized.png
Saved: classification_metrics.csv
Saved: hypothesis_test_results.csv & accuracy_by_group.png
Saved: confidence_calibration.csv & confidence_correlation.png
Saved: analysis_summary.txt
=== STATISTICAL ANALYSIS SUMMARY ===
Dataset Size: 100 valid samples
Overall Accuracy: 0.460

--- Per-Emotion Accuracy ---
ground_truth  accuracy   ci_low  ci_high
     neutral     1.000 0.630583 1.000000
         joy     1.000 0.630583 1.000000
     sadness     0.625 0.244863 0.914767
       anger     0.875 0.473490 0.996840
        fear     0.750 0.349144 0.968146
     disgust     0.250 0.031854 0.650856
    surprise     0.625 0.244863 0.914767
        hate     0.000 0.000000 0.369417
   confusion     0.500 0.157013 0.842987
 frustration     0.125 0.003160 0.526510
     boredom     0.000 0.000000 0.369417
    contempt     0.000 0.00000